In [ ]:
import pandas as pd
df = pd.read_csv("/content/bridge_dataset (1).csv")
print(df.shape)
print(df.columns)




(1340, 15)
Index(['timestamp', 'bridge_id', 'sensor_id', 'acceleration_x',
       'acceleration_y', 'acceleration_z', 'temperature_c', 'humidity_percent',
       'wind_speed_mps', 'fft_peak_freq', 'fft_magnitude', 'degradation_score',
       'structural_condition', 'damage_class', 'forecast_score_next_30d'],
      dtype='object')


In [ ]:

print(df.head())

             timestamp bridge_id sensor_id  acceleration_x  acceleration_y  \
0  2020-01-01 00:00:00      B001        S4       -0.428902        0.009163   
1  2020-01-01 00:15:00      B003        S3        0.086893       -0.005482   
2  2020-01-01 00:30:00      B003        S3       -0.168531       -0.703500   
3  2020-01-01 00:45:00      B002        S3       -0.242926       -0.046838   
4  2020-01-01 01:00:00      B001        S2        0.175638        0.000792   

   acceleration_z  temperature_c  humidity_percent  wind_speed_mps  \
0       -0.448430      24.263205         82.573476        9.129824   
1        0.422973      19.343838         83.545112        6.676185   
2       -0.403903      18.021082         41.881540        1.084121   
3       -0.359685      19.908278         81.787200        0.003722   
4       -0.242574      30.995286         89.394147        1.032235   

   fft_peak_freq  fft_magnitude  degradation_score  structural_condition  \
0       3.264360       1.393159   

## **Map Original Damage Classes to 3 Classes**

In [ ]:
# Define mapping to 3 classes
mapping = {
    'No Damage': 'Normal',
    'Minor': 'Maybe Not Normal',
    'Moderate': 'Not Normal',
    'Severe': 'Not Normal'
}

# Apply mapping
df['damage_class'] = df['damage_class'].map(mapping)

# Check the distribution
print(df['damage_class'].value_counts())


damage_class
Maybe Not Normal    510
Not Normal          464
Normal              366
Name: count, dtype: int64


### **Feature Engineering: Acceleration Magnitude**

In [ ]:
import numpy as np
df['acc_mag'] = np.sqrt(df['acceleration_x']**2 +
                        df['acceleration_y']**2 +
                        df['acceleration_z']**2)
print(df[['acceleration_x','acceleration_y','acceleration_z','acc_mag']].head())


   acceleration_x  acceleration_y  acceleration_z   acc_mag
0       -0.428902        0.009163       -0.448430  0.620588
1        0.086893       -0.005482        0.422973  0.431841
2       -0.168531       -0.703500       -0.403903  0.828524
3       -0.242926       -0.046838       -0.359685  0.436556
4        0.175638        0.000792       -0.242574  0.299485


**Short-term & Long-term RMS for Vibration Analysis**

In [ ]:
# Short-term window
win_short = 8
# Long-term window
win_long = 96
# Short-term RMS
df['rms_short'] = df['acc_mag'].rolling(window=win_short, min_periods=3).apply(lambda x: np.sqrt(np.mean(x**2)), raw=True)
# Long-term RMS
df['rms_long'] = df['acc_mag'].rolling(window=win_long, min_periods=8).apply(lambda x: np.sqrt(np.mean(x**2)), raw=True)
# Drift / long-term change
df['rms_long_diff'] = df['rms_long'] - df['rms_long'].shift(win_long)
print(df[['acc_mag', 'rms_short', 'rms_long', 'rms_long_diff']].head(10))


    acc_mag  rms_short  rms_long  rms_long_diff
0  0.620588        NaN       NaN            NaN
1  0.431841        NaN       NaN            NaN
2  0.828524   0.647577       NaN            NaN
3  0.436556   0.601799       NaN            NaN
4  0.299485   0.554678       NaN            NaN
5  0.450397   0.538702       NaN            NaN
6  0.188737   0.503817       NaN            NaN
7  0.504413   0.503891  0.503891            NaN
8  0.408147   0.476013  0.494170            NaN
9  0.298556   0.463055  0.478223            NaN


In [ ]:
print(df[['acc_mag', 'rms_short', 'rms_long', 'rms_long_diff']].tail(10))

       acc_mag  rms_short  rms_long  rms_long_diff
1330  0.253856   0.379818  0.507458      -0.007739
1331  0.797736   0.420709  0.513929       0.005520
1332  0.622455   0.465144  0.515507       0.008755
1333  0.637325   0.499408  0.518603       0.012499
1334  0.597289   0.537205  0.522055       0.016005
1335  1.074476   0.652714  0.521795       0.008609
1336  0.600915   0.655916  0.522198       0.013849
1337  0.532543   0.675501  0.517897       0.004540
1338  0.429896   0.686548  0.515140       0.004363
1339  0.557315   0.656220  0.515711       0.003873


**Assigining** **Fuzzy Membership**





In [ ]:
import numpy as np

#compute acc_mag z-score per sensor
df['acc_mag_z'] = df.groupby('sensor_id')['acc_mag'].transform(
    lambda x: (x - x.mean()) / x.std()
)
# Initialize membership columns
df['membership_normal'] = 0.0
df['membership_not_normal'] = 0.0

# Assign memberships
#we are clear about these two classes so we will assign membership directly
for i, row in df.iterrows():
    if row['damage_class'] == 'Normal':
        df.at[i, 'membership_normal'] = 1.0
        df.at[i, 'membership_not_normal'] = 0.0
    elif row['damage_class'] == 'Not Normal':
        df.at[i, 'membership_normal'] = 0.0
        df.at[i, 'membership_not_normal'] = 1.0
    else:
#z-score is only used for ambiguous cases
        z = abs(row['acc_mag_z'])
        # Normalize to [0.4,0.6] range for soft membership
        mem_normal = max(0.4, min(0.6, 0.6 - 0.1*z))
        mem_not_normal = 1.0 - mem_normal
        df.at[i, 'membership_normal'] = mem_normal
        df.at[i, 'membership_not_normal'] = mem_not_normal
print(df[['damage_class','acc_mag_z','membership_normal','membership_not_normal']].head(20))


        damage_class  acc_mag_z  membership_normal  membership_not_normal
0         Not Normal   0.827806           0.000000               1.000000
1             Normal  -0.142302           1.000000               0.000000
2         Not Normal   1.822539           0.000000               1.000000
3   Maybe Not Normal  -0.118951           0.588105               0.411895
4   Maybe Not Normal  -0.833654           0.516635               0.483365
5             Normal  -0.234224           1.000000               0.000000
6   Maybe Not Normal  -1.513849           0.448615               0.551385
7   Maybe Not Normal   0.212170           0.578783               0.421217
8         Not Normal  -0.440844           0.000000               1.000000
9   Maybe Not Normal  -0.878702           0.512130               0.487870
10            Normal  -0.681533           1.000000               0.000000
11  Maybe Not Normal  -0.796752           0.520325               0.479675
12        Not Normal   2.241453       

In [ ]:
df.columns

Index(['timestamp', 'bridge_id', 'sensor_id', 'acceleration_x',
       'acceleration_y', 'acceleration_z', 'temperature_c', 'humidity_percent',
       'wind_speed_mps', 'fft_peak_freq', 'fft_magnitude', 'degradation_score',
       'structural_condition', 'damage_class', 'forecast_score_next_30d',
       'acc_mag', 'rms_short', 'rms_long', 'rms_long_diff', 'acc_mag_z',
       'membership_normal', 'membership_not_normal'],
      dtype='object')

**Training and Evaluating a Fuzzy Multi-Class SVM**

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
from sklearn import set_config
set_config(enable_metadata_routing=True)

#  Map damage classes to integers
label_map = {'Normal':0, 'Maybe Not Normal':1, 'Not Normal':2}
y = df['damage_class'].map(label_map)
feature_cols = [
    'acc_mag_z', 'rms_short', 'rms_long', 'rms_long_diff',
    'acc_mag','temperature_c', 'humidity_percent', 'wind_speed_mps',
    'fft_peak_freq', 'fft_magnitude',
    'degradation_score', 'forecast_score_next_30d'
]
X = df[feature_cols].fillna(0)
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Compute fuzzy weights
weights = []
for i, row in df.iterrows():
    if row['damage_class'] == 'Normal':
        weights.append(row['membership_normal'])
    elif row['damage_class'] == 'Maybe Not Normal':
        weights.append(max(row['membership_normal'], row['membership_not_normal']))
    else:
        weights.append(row['membership_not_normal'])
weights = np.array(weights)
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_scaled, y, weights, test_size=0.3, random_state=42
)

#Train fuzzy 3-class SVM
svc = SVC(kernel='rbf', probability=True)
svc.set_fit_request(sample_weight=True)
fsvm = OneVsRestClassifier(svc)
fsvm.fit(X_train, y_train, sample_weight=w_train)
#Evaluation
y_pred = fsvm.predict(X_test)
print(classification_report(
    y_test, y_pred,
    target_names=['Normal','Maybe Not Normal','Not Normal']
))


                  precision    recall  f1-score   support

          Normal       0.81      0.97      0.89       107
Maybe Not Normal       0.97      0.70      0.81       155
      Not Normal       0.86      1.00      0.92       140

        accuracy                           0.88       402
       macro avg       0.88      0.89      0.87       402
    weighted avg       0.89      0.88      0.87       402

